# Phase 2: True SmoothQuant + W8A8 Harness (EXAONE-4.0-1.2B)

이 노트북은 **엔진 수정 없이(weight/config 레벨만)** 다음 2개 실험을 동일 경로로 생성/패키징합니다.

- `Baseline`: W8A8(INT8) only, `embed_tokens`/`lm_head` 보호
- `Variant`: **True SmoothQuant(alpha=0.8) + W8A8(INT8)**

## 전략 요약 (요구 출력 구조)
1️⃣ 현재 상태 분석  
- 기존 `colab_smoothquant_w8a8.ipynb`는 `SmoothQuantModifier`를 import했지만 실제 recipe에는 넣지 않아, 이름-내용 불일치 위험이 있었습니다.

2️⃣ baseline이 0.63 나온 이유 추론  
- 점수 상승의 주 원인은 SmoothQuant 자체보다, `W8A8 INT8 + embed/lm_head 보호` 조합일 가능성이 높습니다.

3️⃣ 위험 요소  
- QK-Reorder-LN 구조에서 Smooth mapping 불일치 시 정확도 하락 가능  
- 과도한 양자화 범위로 PerfNorm 급락 가능  
- 제출 형식(root folder/zip 구조) 불일치 시 제출 실패 가능

4️⃣ 다음 실험 전략 (구체 설정)  
- Baseline: `W8A8`, `targets=["Linear"]`, `ignore=["embed_tokens","lm_head"]`  
- Variant: `recipe[0]=SmoothQuantModifier(alpha=0.8 계열 자동적응)` + `recipe[1]=W8A8`  
- Smooth mapping(regex): `q_proj/k_proj ↔ q_norm/k_norm`, `gate/up/down ↔ post_feedforward_layernorm`  
- 캘리브레이션: `MANTA-1M train`을 **shuffle 후 select**, `N=512`, `max_seq_len=2048`

5️⃣ 예상 점수 변화 방향  
- Baseline 대비 Variant는 PerfNorm 유지/개선 시 Score 상향 가능  
- PerfNorm 손실이 작으면 SpeedNorm 이득이 최종 점수에 직접 반영

6️⃣ SpeedNorm 최적화 전략  
- vLLM 호환 가능한 `W8A8` 경로 우선(GPTQModifier 선호, 실패 시 QuantizationModifier fallback)  
- 동일 엔진 조건에서 per-token time을 짧은 smoke benchmark로 비교

7️⃣ 제출 안정성 체크리스트  
- HF 표준 저장(`save_pretrained(save_compressed=True)` + tokenizer)  
- 기존 노트북과 동일한 `shutil.make_archive(root_dir='.', base_dir=<model_dir>)` 방식 zip  
- zip 크기(<=10GB), 압축해제 크기(<=32GB) 사전 확인


## TASK 0 근거 정리 (기존 노트북 스캔 결과)

아래 3개 노트북의 실제 코드 기준으로 제출 포맷을 맞춥니다.

- `colab_awq_setup_final.ipynb`  
  - `model.save_pretrained(OUT_DIR, save_compressed=True)`  
  - `tokenizer.save_pretrained(OUT_DIR)`  
  - `shutil.make_archive(base_name=zip_name, format="zip", root_dir=".", base_dir=OUT_DIR)`

- `colab_mixed_precision_quantization.ipynb`  
  - `model.save_pretrained(SAVE_DIR, save_compressed=True)`  
  - `tokenizer.save_pretrained(SAVE_DIR)`  
  - `shutil.make_archive(base_name=zip_name, format="zip", root_dir=".", base_dir=SAVE_DIR)`

- `colab_smoothquant_w8a8.ipynb`  
  - `model.save_pretrained(OUT_DIR, save_compressed=True)`  
  - `tokenizer.save_pretrained(OUT_DIR)`  
  - `shutil.make_archive(base_name=zip_name, format="zip", root_dir=".", base_dir=OUT_DIR)`

즉, **저장 폴더 전체를 zip 내부 루트로 포함**하는 포맷입니다. 본 노트북도 동일하게 적용합니다.


In [ ]:
# Optional install cell (manual toggle)
# Heavy reinstall/download must be user-approved.

import subprocess
import sys

INSTALL_DEPS = False  # True로 바꿀 때만 실행

if INSTALL_DEPS:
    pkgs = [
        "torch==2.9.0+cu128",
        "transformers==4.57.3",
        "vllm==0.14.1",
        "datasets",
        "llmcompressor",
        "compressed-tensors",
        "accelerate",
        "sentencepiece",
    ]
    cmd = [sys.executable, "-m", "pip", "install", "-U", *pkgs]
    print("[RUN]", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("[SKIP] INSTALL_DEPS=False. No installation executed.")


In [ ]:
# Version check against competition constraints

from importlib.metadata import PackageNotFoundError, version

REQUIRED = {
    "torch": "2.9.0+cu128",
    "transformers": "4.57.3",
    "vllm": "0.14.1",
}


def get_ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "(not installed)"


def major_minor_patch(ver: str) -> str:
    # torch는 +cu128 suffix가 붙을 수 있어 prefix 비교를 허용
    return ver.split("+")[0]


mismatches = []
for pkg, req in REQUIRED.items():
    cur = get_ver(pkg)
    print(f"{pkg:<13} current={cur} | required={req}")
    if cur == "(not installed)":
        mismatches.append((pkg, req, cur))
    elif pkg == "torch":
        if major_minor_patch(cur) != major_minor_patch(req) or "+cu128" not in cur:
            mismatches.append((pkg, req, cur))
    else:
        if major_minor_patch(cur) != major_minor_patch(req):
            mismatches.append((pkg, req, cur))

if mismatches:
    print()
    print("[사용자 승인 필요] 런타임 버전이 요구 조건과 다릅니다:")
    for m in mismatches:
        print(" -", m)
else:
    print()
    print("[OK] Runtime version check passed.")


In [ ]:
# Experiment configuration (safe defaults + explicit toggles)

import os
from pathlib import Path
from datetime import datetime, timezone

# Path policy: Kaggle 우선, 없으면 현재 작업 디렉터리
WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

SEED = 42
NUM_CALIBRATION_SAMPLES = 512
MAX_SEQUENCE_LENGTH = 2048

# Calibration sampling strategy
# - "shuffle_random": 기존 방식 (shuffle -> select)
# - "mixed_complexity": complexity 기준 상위/중간/랜덤 혼합
CALIB_SAMPLING_MODE = "mixed_complexity"
COMPLEXITY_COLUMN = "complexity_label"
MIX_TOP_RATIO = 0.80
MIX_MID_RATIO = 0.15
MIX_RANDOM_RATIO = 0.05
# 중간 난이도 풀을 정렬 순위 기준으로 자를 구간 (전체의 40~60%)
MID_COMPLEXITY_RANGE = (0.40, 0.60)

SCHEME = "W8A8"
TARGETS = ["Linear"]
BASE_IGNORE = ["embed_tokens", "lm_head"]

# Optional edge-layer protection toggle
PROTECT_EDGE_LAYERS = False
EDGE_LAYERS = ["model.layers.0", "model.layers.29"]

# Main run toggles
RUN_BASELINE = True
RUN_VARIANT = True
RUN_SMOKE_TEST = False

# Optional alpha grid (default OFF)
ENABLE_GRID = False
ALPHA_GRID = [0.6, 0.7, 0.8, 0.9]

# vLLM smoke test config (very small)
SMOKE_PROMPTS = [
    "양자화된 EXAONE 모델의 장점을 간단히 설명해줘.",
    "Explain one tradeoff of INT8 quantization.",
]
SMOKE_MAX_TOKENS = 64
SMOKE_TEMPERATURE = 0.0

print("WORKDIR:", WORKDIR)
print("MODEL_ID:", MODEL_ID)
print("Calibration:", NUM_CALIBRATION_SAMPLES, "samples /", MAX_SEQUENCE_LENGTH, "tokens")
print("CALIB_SAMPLING_MODE:", CALIB_SAMPLING_MODE)
if CALIB_SAMPLING_MODE == "mixed_complexity":
    print(
        "MIX ratios(top/mid/random)=",
        f"{MIX_TOP_RATIO:.2f}/{MIX_MID_RATIO:.2f}/{MIX_RANDOM_RATIO:.2f}",
        "| complexity_col=",
        COMPLEXITY_COLUMN,
        "| mid_range=",
        MID_COMPLEXITY_RANGE,
    )
print("RUN_BASELINE:", RUN_BASELINE, "RUN_VARIANT:", RUN_VARIANT, "ENABLE_GRID:", ENABLE_GRID)


In [ ]:
# Core imports

import json
import os
import re
import shutil
import subprocess
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot


In [ ]:
# Helpers: safety checks, modifiers, dataset prep, packaging


def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def maybe_hf_token() -> Optional[str]:
    token = os.environ.get("HF_TOKEN")
    if token:
        print("[INFO] HF_TOKEN detected in environment (value is hidden).")
    else:
        print("[INFO] HF_TOKEN not set. Public model/dataset access만 가능.")
    return token


def ensure_no_smoothquant_monkeypatch() -> None:
    # Abort if known early-return patch marker is detected in installed llmcompressor.
    import llmcompressor

    sq_base = Path(llmcompressor.__file__).resolve().parent / "modifiers" / "smoothquant" / "base.py"
    txt = sq_base.read_text(encoding="utf-8")
    suspicious_markers = [
        "[PATCH] disable smoothquant hook",
        "disable smoothquant hook",
    ]
    for marker in suspicious_markers:
        if marker in txt:
            raise RuntimeError(
                f"Detected SmoothQuant monkeypatch marker in {sq_base}: {marker}. "
                "This notebook forbids SmoothQuant early-return patching."
            )
    print(f"[OK] No known SmoothQuant monkeypatch marker found: {sq_base}")


def build_ignore_list() -> List[str]:
    ignore = list(BASE_IGNORE)
    if PROTECT_EDGE_LAYERS:
        ignore.extend(EDGE_LAYERS)
    return ignore


def build_layer_mappings(model) -> List[Any]:
    from llmcompressor.modifiers.smoothquant.utils import LayerMap

    checks = {
        "q_proj": r"self_attn\.q_proj$",
        "q_norm": r"self_attn\.q_norm$",
        "k_proj": r"self_attn\.k_proj$",
        "k_norm": r"self_attn\.k_norm$",
        "post_ffn_ln": r"post_feedforward_layernorm$",
        "down_proj": r"down_proj$",
    }
    for name, patt in checks.items():
        cnt = sum(1 for n, _ in model.named_modules() if re.search(patt, n))
        print(f"[CHECK] {name:<12}: {cnt}")

    return [
        LayerMap(
            balance_layers=[r"re:.*self_attn\.q_proj$"],
            smooth_layers=r"re:.*self_attn\.q_norm$",
        ),
        LayerMap(
            balance_layers=[r"re:.*self_attn\.k_proj$"],
            smooth_layers=r"re:.*self_attn\.k_norm$",
        ),
        LayerMap(
            balance_layers=[r"re:.*gate_proj$", r"re:.*up_proj$", r"re:.*down_proj$"],
            smooth_layers=r"re:.*post_feedforward_layernorm$",
        ),
    ]


def build_smoothquant_modifier(alpha: float, mappings: List[Any]) -> Any:
    from llmcompressor.modifiers.smoothquant import SmoothQuantModifier

    attempts = []
    for key in ("alpha", "smoothing_strength", "strength"):
        kwargs = {key: alpha, "mappings": mappings}
        attempts.append(kwargs)
        try:
            mod = SmoothQuantModifier(**kwargs)
            print(f"[OK] SmoothQuantModifier created with kwargs={kwargs}")
            return mod
        except TypeError:
            continue

    # Fallback: create with mappings and set attribute dynamically
    mod = SmoothQuantModifier(mappings=mappings)
    for attr in ("alpha", "smoothing_strength", "strength"):
        if hasattr(mod, attr):
            setattr(mod, attr, alpha)
            print(f"[OK] SmoothQuantModifier created via fallback, set {attr}={alpha}")
            return mod

    raise RuntimeError(f"SmoothQuant alpha adaptation failed. Tried: {attempts}")


def build_w8a8_modifier(ignore: List[str]) -> Any:
    # Prefer GPTQModifier(scheme='W8A8') for vLLM compatibility; fallback to QuantizationModifier.
    try:
        from llmcompressor.modifiers.quantization import GPTQModifier

        mod = GPTQModifier(
            scheme=SCHEME,
            targets=TARGETS,
            ignore=ignore,
        )
        print("[OK] Using GPTQModifier for W8A8")
        return mod
    except Exception as gptq_err:
        print(f"[WARN] GPTQModifier W8A8 unavailable: {gptq_err}")

    from llmcompressor.modifiers.quantization import QuantizationModifier

    mod = QuantizationModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore,
    )
    print("[OK] Using QuantizationModifier for W8A8 (fallback)")
    return mod


def preprocess_fn(example: Dict[str, Any], tokenizer) -> Dict[str, str]:
    if "conversations" in example:
        txt = tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False,
        )
        return {"text": txt}

    # Fallback to common text fields
    for key in ("text", "prompt", "content", "question"):
        if key in example and isinstance(example[key], str):
            return {"text": example[key]}
    return {"text": json.dumps(example, ensure_ascii=False)}


def sample_mixed_by_complexity(ds, n_samples: int, seed: int):
    """Select calibration samples with 80/15/5 mix by complexity rank.

    - top 80%: highest complexity
    - mid 15%: middle-complexity band (MID_COMPLEXITY_RANGE)
    - random 5%: random from remaining pool
    """
    from datasets import concatenate_datasets

    total_len = len(ds)
    if total_len == 0:
        raise RuntimeError("Dataset is empty.")

    if COMPLEXITY_COLUMN not in ds.column_names:
        print(
            f"[WARN] '{COMPLEXITY_COLUMN}' not found. "
            "Fallback to shuffle_random sampling."
        )
        return ds.shuffle(seed=seed).select(range(min(n_samples, total_len))), {
            "mode": "shuffle_random_fallback",
            "top": 0,
            "mid": 0,
            "random": min(n_samples, total_len),
            "total": min(n_samples, total_len),
        }

    target_n = min(n_samples, total_len)
    n_top = int(target_n * MIX_TOP_RATIO)
    n_mid = int(target_n * MIX_MID_RATIO)
    n_rand = target_n - n_top - n_mid

    ds_idx = ds.add_column("__row_id__", list(range(total_len)))

    # Complexity descending sort
    ds_sorted = ds_idx.sort(COMPLEXITY_COLUMN, reverse=True)

    # Top slice
    top_n = min(n_top, len(ds_sorted))
    top_ds = ds_sorted.select(range(top_n)) if top_n > 0 else ds_sorted.select([])

    # Mid slice pool by percentile range
    low_q, high_q = MID_COMPLEXITY_RANGE
    low_idx = max(0, int(total_len * low_q))
    high_idx = min(total_len, int(total_len * high_q))
    if high_idx <= low_idx:
        low_idx = total_len // 3
        high_idx = (total_len * 2) // 3

    mid_pool = ds_sorted.select(range(low_idx, high_idx)) if high_idx > low_idx else ds_sorted.select([])
    mid_n = min(n_mid, len(mid_pool))
    mid_ds = mid_pool.shuffle(seed=seed + 1).select(range(mid_n)) if mid_n > 0 else ds_sorted.select([])

    # Random slice from remaining indices (no overlap)
    used = set(top_ds["__row_id__"]) | set(mid_ds["__row_id__"])
    remaining_indices = [i for i in range(total_len) if i not in used]

    if n_rand > 0 and remaining_indices:
        import random

        rng = random.Random(seed + 2)
        if n_rand < len(remaining_indices):
            rand_indices = rng.sample(remaining_indices, n_rand)
        else:
            rand_indices = remaining_indices
        rand_ds = ds_idx.select(rand_indices)
    else:
        rand_ds = ds_sorted.select([])

    parts = [x for x in [top_ds, mid_ds, rand_ds] if len(x) > 0]
    mixed = concatenate_datasets(parts) if parts else ds_sorted.select([])
    mixed = mixed.shuffle(seed=seed + 3)

    # Safety fill if rounding/overlap caused shortage
    if len(mixed) < target_n:
        used2 = set(mixed["__row_id__"])
        fill_candidates = [i for i in range(total_len) if i not in used2]
        need = min(target_n - len(mixed), len(fill_candidates))
        if need > 0:
            import random

            rng = random.Random(seed + 4)
            fill_idx = rng.sample(fill_candidates, need)
            fill_ds = ds_idx.select(fill_idx)
            mixed = concatenate_datasets([mixed, fill_ds]).shuffle(seed=seed + 5)

    # Drop helper column
    mixed = mixed.remove_columns(["__row_id__"])

    info = {
        "mode": "mixed_complexity",
        "top": len(top_ds),
        "mid": len(mid_ds),
        "random": len(rand_ds),
        "total": len(mixed),
        "top_ratio": (len(top_ds) / max(len(mixed), 1)),
        "mid_ratio": (len(mid_ds) / max(len(mixed), 1)),
        "random_ratio": (len(rand_ds) / max(len(mixed), 1)),
    }
    return mixed, info


def prepare_calibration_dataset(tokenizer):
    print(f"[INFO] Loading dataset: {DATASET_ID} ({DATASET_SPLIT})")
    ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)

    n = min(NUM_CALIBRATION_SAMPLES, len(ds))

    if CALIB_SAMPLING_MODE == "mixed_complexity":
        ds, info = sample_mixed_by_complexity(ds, n_samples=n, seed=SEED)
        print(
            "[INFO] Sampling summary:",
            f"top={info['top']} ({info['top_ratio']:.3f}),",
            f"mid={info['mid']} ({info['mid_ratio']:.3f}),",
            f"random={info['random']} ({info['random_ratio']:.3f}),",
            f"total={info['total']}",
        )
    elif CALIB_SAMPLING_MODE == "shuffle_random":
        ds = ds.shuffle(seed=SEED).select(range(n))
        print(f"[INFO] Sampling summary: mode=shuffle_random total={len(ds)}")
    else:
        raise ValueError(
            f"Unsupported CALIB_SAMPLING_MODE={CALIB_SAMPLING_MODE}. "
            "Use 'mixed_complexity' or 'shuffle_random'."
        )

    cols = ds.column_names
    ds = ds.map(lambda x: preprocess_fn(x, tokenizer), remove_columns=cols)

    print(f"[INFO] Calibration Dataset Ready: {len(ds)} samples")
    return ds


def directory_size_bytes(path: Path) -> int:
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total


def check_required_hf_files(model_dir: Path) -> None:
    required = ["config.json", "tokenizer.json", "tokenizer_config.json"]
    print("[CHECK] Required HF files")
    for name in required:
        p = model_dir / name
        print(f" - {name}:", p.exists())


def print_dir_report(path: Path, max_files: int = 80) -> None:
    print()
    print("[REPORT] Directory:", path)
    if not path.exists():
        print(" - not found")
        return

    size_bytes = directory_size_bytes(path)
    print(f" - size bytes: {size_bytes:,}")
    print(f" - size GB   : {size_bytes / (1024**3):.4f}")

    files = sorted([p for p in path.rglob("*") if p.is_file()])
    print(f" - file count: {len(files)}")
    for p in files[:max_files]:
        rel = p.relative_to(path)
        print("   ", rel)
    if len(files) > max_files:
        print(f"   ... ({len(files)-max_files} more)")

    check_required_hf_files(path)

    # Shell du summary if available
    try:
        out = subprocess.check_output(["du", "-sh", str(path)], text=True).strip()
        print(" - du -sh    :", out)
    except Exception as exc:
        print(" - du -sh    : unavailable", exc)


def build_zip_from_model_dir(model_dir: Path, tag: str) -> Path:
    stamp = utc_stamp()
    zip_base = WORKDIR / f"{tag}_{stamp}_submit"

    model_dir_rel = os.path.relpath(model_dir, start=WORKDIR)

    # Keep same packaging style as existing notebooks:
    # make_archive(base_name=..., format='zip', root_dir='.', base_dir=OUT_DIR)
    cwd_prev = Path.cwd()
    os.chdir(WORKDIR)
    try:
        shutil.make_archive(
            base_name=str(zip_base),
            format="zip",
            root_dir=".",
            base_dir=model_dir_rel,
        )
    finally:
        os.chdir(cwd_prev)

    zip_path = zip_base.with_suffix(".zip")
    zip_size = zip_path.stat().st_size
    uncompressed_size = directory_size_bytes(model_dir)

    print()
    print("[REPORT] ZIP:", zip_path)
    print(f" - zip size bytes        : {zip_size:,}")
    print(f" - zip size GB           : {zip_size / (1024**3):.4f}")
    print(f" - uncompressed size GB  : {uncompressed_size / (1024**3):.4f}")
    print(f" - <=10GB zip            : {zip_size <= 10 * (1024**3)}")
    print(f" - <=32GB uncompressed   : {uncompressed_size <= 32 * (1024**3)}")

    return zip_path


In [ ]:
# Single experiment runner: baseline OR true smoothquant variant


def run_experiment(tag: str, enable_smooth: bool, alpha: float = 0.8) -> Dict[str, Any]:
    ensure_no_smoothquant_monkeypatch()
    token = maybe_hf_token()

    print()
    print("=" * 90)
    print(f"[RUN] tag={tag} | smooth={enable_smooth} | alpha={alpha}")
    print("=" * 90)

    model_kwargs = {
        "torch_dtype": torch.bfloat16,
        "trust_remote_code": True,
        "device_map": "auto",
    }
    if token:
        model_kwargs["token"] = token

    tok_kwargs = {"trust_remote_code": True}
    if token:
        tok_kwargs["token"] = token

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **tok_kwargs)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

    tie_embed = getattr(model.config, "tie_word_embeddings", None)
    print("[INFO] tie_word_embeddings:", tie_embed)

    ds = prepare_calibration_dataset(tokenizer)

    ignore = build_ignore_list()
    print("[INFO] ignore modules:", ignore)

    recipe = []
    if enable_smooth:
        mappings = build_layer_mappings(model)
        sq_mod = build_smoothquant_modifier(alpha=alpha, mappings=mappings)
        recipe.append(sq_mod)  # must be recipe[0]

    q_mod = build_w8a8_modifier(ignore=ignore)
    recipe.append(q_mod)

    print("[INFO] recipe order:", [type(x).__name__ for x in recipe])

    oneshot(
        model=model,
        dataset=ds,
        recipe=recipe,
        max_seq_length=MAX_SEQUENCE_LENGTH,
        num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    )

    out_dir = WORKDIR / f"model_{tag}"
    out_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(out_dir, save_compressed=True)
    tokenizer.save_pretrained(out_dir)

    print_dir_report(out_dir)
    zip_path = build_zip_from_model_dir(out_dir, tag=tag)

    return {
        "tag": tag,
        "smooth": enable_smooth,
        "alpha": alpha,
        "model_dir": str(out_dir),
        "zip_path": str(zip_path),
    }


In [ ]:
# Execute baseline + variant (+ optional alpha grid)

run_summaries = []

if RUN_BASELINE:
    run_summaries.append(
        run_experiment(
            tag="phase2_baseline_w8a8",
            enable_smooth=False,
            alpha=0.8,
        )
    )

if RUN_VARIANT:
    run_summaries.append(
        run_experiment(
            tag="phase2_true_smoothquant_a0.8_w8a8",
            enable_smooth=True,
            alpha=0.8,
        )
    )

if ENABLE_GRID:
    for a in ALPHA_GRID:
        run_summaries.append(
            run_experiment(
                tag=f"phase2_true_smoothquant_a{a:.1f}_w8a8",
                enable_smooth=True,
                alpha=float(a),
            )
        )

print()
print("[SUMMARY] Generated artifacts")
for row in run_summaries:
    print(row)


In [ ]:
# Optional vLLM smoke test (very small) - no engine patching

if not RUN_SMOKE_TEST:
    print("[SKIP] RUN_SMOKE_TEST=False")
else:
    from vllm import LLM, SamplingParams

    def rough_per_token_time(model_path: str, prompts: List[str], max_tokens: int = 64) -> Dict[str, float]:
        llm = LLM(
            model=model_path,
            trust_remote_code=True,
            tensor_parallel_size=1,
            gpu_memory_utilization=0.85,
        )
        sp = SamplingParams(max_tokens=max_tokens, temperature=SMOKE_TEMPERATURE)

        t0 = time.time()
        outputs = llm.generate(prompts, sp)
        elapsed = time.time() - t0

        gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        tpt = elapsed / max(gen_tokens, 1)

        return {
            "elapsed_sec": elapsed,
            "generated_tokens": gen_tokens,
            "time_per_token_sec": tpt,
            "tokens_per_sec": (gen_tokens / elapsed) if elapsed > 0 else 0.0,
        }

    smoke_rows = []
    for row in run_summaries:
        stats = rough_per_token_time(
            model_path=row["model_dir"],
            prompts=SMOKE_PROMPTS,
            max_tokens=SMOKE_MAX_TOKENS,
        )
        out = {"tag": row["tag"], **stats}
        smoke_rows.append(out)
        print(out)

    # Baseline vs variant quick ratio (if both present)
    base = next((x for x in smoke_rows if "baseline" in x["tag"]), None)
    var = next((x for x in smoke_rows if "smoothquant" in x["tag"]), None)
    if base and var:
        speed_norm_proxy = 1.0 - (var["time_per_token_sec"] / base["time_per_token_sec"])
        print()
        print("[SMOKE] rough SpeedNorm proxy:", speed_norm_proxy)


## 제출 안정성 체크 포인트
- 엔진/vLLM/CUDA 커널 수정 코드 없음
- SmoothQuant early-return 패치 사용 금지(실행 시 marker 검사)
- baseline/variant 모두 HF 표준 저장 + tokenizer 저장
- 기존 노트북과 동일한 zip 생성 방식(`make_archive` with model directory as base_dir)
- zip 및 압축해제 크기 제한(10GB/32GB) 사전 출력
